# 02 — Preprocessing & Frame Labeling

Goals:
1. Load the raw corpus CSV exported from BigQuery
2. Run the full preprocessing pipeline (dates, dedup, tone parsing, country, frame flags)
3. Inspect frame assignments and spot-check a sample from each frame
4. Save the preprocessed corpus to `data/interim/`

In [ ]:
from pathlib import Path
import pandas as pd
import sys

sys.path.insert(0, str(Path('..')))

from src.preprocessing import run_pipeline, load_raw
from src.dictionaries import FRAME_DICTS, FRAME_COLS, MILESTONES

## 1. Load raw data

In [ ]:
RAW_PATH = Path('../data/raw/gdelt_genai_gov.csv')
raw_df = load_raw(RAW_PATH)
print(f'Loaded {len(raw_df):,} rows, {raw_df.shape[1]} columns')
raw_df.head(2)

## 2. Run preprocessing pipeline

In [ ]:
df = run_pipeline(raw_df)
print(f'After dedup: {len(df):,} rows')
df.dtypes

## 3. Frame distribution overview

In [ ]:
import matplotlib.pyplot as plt

frame_counts = df[FRAME_COLS].sum().rename(index=lambda x: x.replace('frame_', ''))
frame_counts.sort_values(ascending=True).plot(kind='barh', figsize=(8, 4), title='Total keyword hits per frame')
plt.tight_layout()
plt.show()

In [ ]:
# Dominant frame distribution
df['dominant_frame'].value_counts(dropna=False)

## 4. Manual spot-check — sample 5 articles per dominant frame

In [ ]:
SPOT_COLS = ['DATE', 'SourceCommonName', 'dominant_frame', 'Quotations']

for frame_name in FRAME_DICTS:
    sample = df[df['dominant_frame'] == frame_name][SPOT_COLS].sample(
        min(5, (df['dominant_frame'] == frame_name).sum()), random_state=42
    )
    print(f'\n=== {frame_name} (n={len(df[df["dominant_frame"]==frame_name]):,}) ===')
    for _, row in sample.iterrows():
        print(f'  [{row["DATE"]}] {row["SourceCommonName"]}')
        print(f'  {str(row["Quotations"])[:200]}')

## 5. Event window coverage check

In [ ]:
df[['event_name', 'rel_month']].value_counts(dropna=True).sort_index()

## 6. Save preprocessed corpus

In [ ]:
OUT_PATH = Path('../data/interim/gdelt_preprocessed.parquet')
df.to_parquet(OUT_PATH, index=False)
print(f'Saved to {OUT_PATH}')